# 04 — SIL Streaming Simulation
# Giai đoạn 3 — Mục 3.3 — Mô phỏng xử lý luồng tín hiệu
**Mục đích**: minh họa pipeline xử lý liên tục từ tín hiệu đến chẩn đoán.

In [1]:
from pathlib import Path
import numpy as np
import time
import pandas as pd
import json
import joblib
import tensorflow as tf
from scipy.signal import resample_poly

- Giả định features_full chứa hàm extract_features_for_window (xuất ra 32 đặc trưng)

In [2]:
from common import io_utils, features_full, pipeline, config as cfg

OUTPUT_DIR = Path("./outputs")
MODELS_DIR = OUTPUT_DIR / "models"
TABLES_DIR = OUTPUT_DIR / "tables"

- 1. Đọc dữ liệu sạch và cấu hình động

In [3]:
manifest = pd.read_csv(TABLES_DIR / "manifest_clean.csv")

with open(TABLES_DIR / "bandpass_config.json") as f:
    bandpass_cfg = json.load(f)
BAND_HZ = tuple(bandpass_cfg["band_hz"])
LP_CUTOFF = bandpass_cfg["lp_cutoff_hz"]

WINDOW_SIZE = 2048
STRIDE = 1024
TARGET_FS = 12000

FileNotFoundError: [Errno 2] No such file or directory: 'outputs\\tables\\manifest_clean.csv'

- Khởi tạo TFLite Interpreter và Scaler

In [ ]:
model_path = MODELS_DIR / "mlp_model.tflite" 
scaler_path = MODELS_DIR / "mlp_scaler.pkl"

if not model_path.exists() or not scaler_path.exists():
    raise FileNotFoundError("Chưa tìm thấy mô hình hoặc scaler. Hãy kiểm tra lại thư mục models/")

interpreter = tf.lite.Interpreter(model_path=str(model_path))
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

scaler = joblib.load(scaler_path)

- 3. Lấy dữ liệu mẫu và chuẩn hóa tần số (Resample)

In [ ]:
fp = pipeline.pick_file(manifest, label="OR", load_hp=0)
fs_map = dict(zip(manifest['file_path'], manifest['fs']))
current_fs = fs_map.get(str(fp), TARGET_FS)

signal_raw = io_utils.load_de_signal(Path(fp))
if current_fs != TARGET_FS:
    signal = resample_poly(signal_raw, TARGET_FS, current_fs)
else:
    signal = signal_raw

- 4. Mô phỏng SIL Streaming (Xử lý tín hiệu thời gian thực)

In [ ]:
print(f"Bắt đầu mô phỏng SIL cho file: {Path(fp).name}")
print("="*60)

num_windows = (len(signal) - WINDOW_SIZE) // STRIDE + 1
latency_list = []
predictions = []

for i in range(num_windows):
    start_idx = i * STRIDE
    end_idx = start_idx + WINDOW_SIZE
    window_data = signal[start_idx:end_idx]
    
    # Bắt đầu bấm giờ đo trễ (Latency)
    t0 = time.perf_counter()
    
    # a. Rút trích 32 đặc trưng
    feat_vector = features_full.extract_features_for_window(
        window_data, 
        fs=TARGET_FS, 
        band_hz=BAND_HZ, 
        lp_cutoff=LP_CUTOFF
    )
    
    # b. Chuẩn hóa dữ liệu (Scaler yêu cầu đầu vào 2D)
    feat_scaled = scaler.transform([feat_vector]).astype(np.float32)
    
    # c. Suy luận bằng TFLite
    interpreter.set_tensor(input_details[0]['index'], feat_scaled)
    interpreter.invoke()
    output_data = interpreter.get_tensor(output_details[0]['index'])
    
    # d. Xử lý kết quả (Giả định đầu ra là Softmax)
    pred_class = np.argmax(output_data)
    confidence = np.max(output_data)
    
    t1 = time.perf_counter()
    process_time_ms = (t1 - t0) * 1000
    latency_list.append(process_time_ms)
    predictions.append(pred_class)

In [ ]:
if i % 10 == 0:
        print(f"Khung {i:03d} | Trễ: {process_time_ms:.2f} ms | Dự đoán Class: {pred_class} (Confidence: {confidence:.2f})")

print("="*60)
print(f"Tổng số khung xử lý: {num_windows}")
print(f"Độ trễ trung bình mỗi khung (Mô phỏng PC): {np.mean(latency_list):.2f} ms")